### Glücksrad – Lösung Schritt für Schritt

*30. Bundeswettbewerb Informatik (2011), 1. Runde, Junioraufgabe 2*

Um den Millionengewinn bei einer großen Fernsehshow zu vergeben, wurde ein Glücksrad gebaut. Das Glücksrad hat 6 Felder und
wird immer gegen den Uhrzeigersinn in Gang gesetzt.

Bei der Konstruktion wurden die Wahrscheinlichkeiten festgelegt, mit denen sich das Glücksrad um eine bestimmte Anzahl Felder
weiter bewegt:

| Anzahl Felder | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| Wahrscheinlichkeit | 5/15 | 4/15 | 3/15 | 2/15 | 1/15 | 0/15 |

Den Millionengewinn erhält ein Teilnehmer, wenn er es schafft, bei sechsmaligem Drehen jedes Feld genau ein Mal zu erreichen.
Beispielsweise bringt die Folge (A)–C–D–B–E–A–F den Millionengewinn. Sie hat Wahrscheinlichkeit
4/15 · 5/15 · 2/15 · 3/15 · 4/15 · 1/15 = 32/759375 ≈ 0,00004. Aber natürlich gibt es viele andere Folgen, die auch den
Millionengewinn bringen. Wie groß ist die Wahrscheinlichkeit, den Millionengewinn zu erhalten?

#### Junioraufgabe 2

Schreibe ein Programm, das für eine beliebig gegebene Wahrscheinlichkeitstabelle die Gewinnwahrscheinlichkeit errechnet.
Das Programm soll für Glücksräder mit bis zu zehn Feldern arbeiten können.

Beispieldaten gibt es zu dieser Aufgabe nicht.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Wahrscheinlichkeit einer Folge:** Die Wahrscheinlichkeiten der einzelnen Drehungen werden **multipliziert**. Alle Folgen, die zum Gewinn führen, sind verschiedene Möglichkeiten, deren Wahrscheinlichkeiten man **addiert**.
- **Rechnen im Kreis (Modulo):** Hinter dem letzten Feld kommt wieder das erste. Von Feld `pos` aus erreicht man mit `k` Feldern Weiterdrehen das Feld `(pos + k) % n`.
- **Rekursion mit Zurücknehmen (Backtracking):** Wir probieren Drehung für Drehung alle Möglichkeiten aus. Die Funktion ruft sich für die nächste Drehung selbst auf. Danach nimmt man die Entscheidung zurück und probiert die nächste Möglichkeit.
- **Alle Möglichkeiten (Brute Force) und die Zahl der Folgen:** Es gibt höchstens `n!` Folgen (bei 10 Feldern 3 628 800), das ist mit Rekursion noch machbar, aber langsam.
- **Merken von Zwischenergebnissen (Memoization, dynamische Programmierung):** Viele Wege führen zur selben Situation (selbe besuchte Felder, selbes aktuelles Feld). Für jede solche Situation rechnet man nur **einmal** und merkt sich das Ergebnis in einem Wörterbuch (`dict`).
  Das macht das Programm viel schneller (es gibt nur `2^n · n` Situationen).

**Nützliche Funktionen und Bibliotheken**
- `%`, Listen und `list.copy`, rekursive Funktionen, `dict`, `tuple`
- `fractions.Fraction` für exakte Brüche (statt Kommazahlen mit Rundungsfehlern), `math.factorial` für `n!`

## Die Idee

Wir gehen die Drehungen der Reihe nach durch. Am Anfang steht das Rad auf Feld 0 (das ist Feld A). Dieses Feld zählt **noch nicht** als besucht. Der Spieler hat gewonnen, wenn nach `n` Drehungen jedes der `n` Felder genau einmal erreicht wurde.
Das Startfeld A wird also später erreicht (zum Beispiel beim fünften Dreh), das ist erlaubt.

Rekursive Funktion `berechne(besucht, pos)`:

- Sind alle Felder besucht, haben wir gewonnen: das Ergebnis ist `1` (die Wahrscheinlichkeit, jetzt noch zu gewinnen, ist 1).
- Sonst: Für jede Schrittweite `k` von 1 bis `n` mit der Wahrscheinlichkeit `p[k-1]` berechnen wir das Feld `neu = (pos + k) % n`. Ist es noch nicht besucht, markieren wir es als besucht, rechnen weiter und addieren
  `p[k-1] * berechne(...)`. Danach nehmen wir die Markierung wieder zurück.

Das Ergebnis ist die Summe über alle Wege, wobei jeder Weg mit den Wahrscheinlichkeiten seiner Schritte multipliziert wird. Genau so, wie die Aufgabe es beschreibt.

## Schritt 1: Die einfache rekursive Lösung

Die Liste `p` enthält die Wahrscheinlichkeiten: `p[0]` ist die Wahrscheinlichkeit, 1 Feld weiterzudrehen, `p[1]` für 2 Felder und so weiter. Die Anzahl der Felder ist `n = len(p)`.
`besucht` ist eine Liste aus `True` und `False` mit einem Eintrag je Feld.

Wir rechnen mit `Fraction` (Brüche) statt mit Kommazahlen: Dann ist das Ergebnis **exakt**, und wir können es zum Schluss als Bruch und als Dezimalzahl ausgeben. `Fraction(5, 15)` ist der Bruch 5/15, den Python automatisch auf 1/3 kürzt.

In [ ]:
from fractions import Fraction


def berechne(p, besucht, pos, anzahl):
    n = len(p)
    if anzahl == n:
        return 1                                    # alle Felder besucht: gewonnen
    summe = 0
    for k in range(1, n + 1):
        neu = (pos + k) % n
        if not besucht[neu]:
            besucht[neu] = True                     # Feld besuchen
            summe = summe + p[k - 1] * berechne(p, besucht, neu, anzahl + 1)
            besucht[neu] = False                    # und wieder zurücknehmen
    return summe


def gewinn_einfach(p):
    n = len(p)
    besucht = [False] * n
    return berechne(p, besucht, 0, 0)


rad = [Fraction(5, 15), Fraction(4, 15), Fraction(3, 15), Fraction(2, 15), Fraction(1, 15), Fraction(0, 15)]
ergebnis = gewinn_einfach(rad)
print('Gewinnwahrscheinlichkeit:', ergebnis, '=', float(ergebnis))

Gewinnwahrscheinlichkeit: 796/16875 = 0.04717037037037037


Für das Rad aus der Aufgabenstellung ergibt sich die Wahrscheinlichkeit 796/16875 ≈ **0,04717**. Das passt zu den Lösungshinweisen (0,04717).

Das Beispiel aus der Aufgabe (die Folge (A)–C–D–B–E–A–F mit der Wahrscheinlichkeit 32/759375) ist nur **einer** der Wege, die zum Gewinn führen. Wir rechnen ihn nach:

In [ ]:
def wahrscheinlichkeit_der_folge(p, felder):
    n = len(p)
    pos = 0
    wk = 1
    for ziel in felder:
        k = (ziel - pos) % n
        if k == 0:
            k = n                       # um das ganze Rad: n Felder weiter
        wk = wk * p[k - 1]
        pos = ziel
    return wk


# (A)-C-D-B-E-A-F mit A = 0, B = 1, ... F = 5
folge = [2, 3, 1, 4, 0, 5]
print(wahrscheinlichkeit_der_folge(rad, folge))

32/759375


Das ist der Bruch 32/759375 aus der Aufgabenstellung. Die ganze Gewinnwahrscheinlichkeit ist die Summe der Wahrscheinlichkeiten aller solcher Folgen. Das Programm zählt sie nicht einzeln auf, sondern addiert sie beim Durchprobieren.

## Schritt 2: Wie lange dauert das?

Für ein Rad mit `n` Feldern gibt es bis zu `n!` Folgen, die jedes Feld genau einmal besuchen:

| n | 4 | 6 | 8 | 9 | 10 |
|---|---|---|---|---|---|
| n! | 24 | 720 | 40 320 | 362 880 | 3 628 800 |

Die einfache Lösung ist also für 10 Felder ziemlich langsam (mehrere Millionen Aufrufe). Wir probieren sie bis 8 Felder aus. Für 10 Felder nehmen wir die schnellere Lösung aus Schritt 3.

In [ ]:
import math

for n in range(1, 9):
    gleich_verteilt = [Fraction(1, n)] * n
    ergebnis = gewinn_einfach(gleich_verteilt)
    print(f'{n} Felder, jede Weite mit Wahrscheinlichkeit 1/{n}: {float(ergebnis):.6f}   (n! = {math.factorial(n)})')

1 Felder, jede Weite mit Wahrscheinlichkeit 1/1: 1.000000   (n! = 1)
2 Felder, jede Weite mit Wahrscheinlichkeit 1/2: 0.500000   (n! = 2)
3 Felder, jede Weite mit Wahrscheinlichkeit 1/3: 0.222222   (n! = 6)
4 Felder, jede Weite mit Wahrscheinlichkeit 1/4: 0.093750   (n! = 24)
5 Felder, jede Weite mit Wahrscheinlichkeit 1/5: 0.038400   (n! = 120)
6 Felder, jede Weite mit Wahrscheinlichkeit 1/6: 0.015432   (n! = 720)
7 Felder, jede Weite mit Wahrscheinlichkeit 1/7: 0.006120   (n! = 5040)
8 Felder, jede Weite mit Wahrscheinlichkeit 1/8: 0.002403   (n! = 40320)


## Schritt 3: Zwischenergebnisse merken

Viele verschiedene Wege führen zu derselben **Situation**: Es sind dieselben Felder besucht, und das Rad steht auf demselben Feld. Wie es jetzt weitergeht, hängt nur von dieser Situation ab, nicht davon, auf welchem Weg man sie erreicht hat.
Wir merken uns deshalb das Ergebnis jeder Situation in einem Wörterbuch `gemerkt`. Der Schlüssel ist ein Paar aus der Liste der besuchten Felder (als `tuple`, denn nur unveränderliche Werte können Schlüssel sein) und der Position.
Kommt dieselbe Situation noch einmal vor, nehmen wir das gemerkte Ergebnis.

Es gibt höchstens `2ⁿ · n` Situationen (für n = 10 sind das 10 240) statt `n!` Wege, daher ist das Programm auch für 10 Felder blitzschnell.

In [ ]:
def berechne_gemerkt(p, besucht, pos, anzahl, gemerkt):
    n = len(p)
    if anzahl == n:
        return 1
    schluessel = (tuple(besucht), pos)
    if schluessel in gemerkt:
        return gemerkt[schluessel]                  # diese Situation haben wir schon berechnet
    summe = 0
    for k in range(1, n + 1):
        neu = (pos + k) % n
        if not besucht[neu]:
            besucht[neu] = True
            summe = summe + p[k - 1] * berechne_gemerkt(p, besucht, neu, anzahl + 1, gemerkt)
            besucht[neu] = False
    gemerkt[schluessel] = summe
    return summe


def gewinn(p):
    n = len(p)
    return berechne_gemerkt(p, [False] * n, 0, 0, {})


print('Rad aus der Aufgabe:', gewinn(rad), '=', float(gewinn(rad)))
for n in range(1, 9):
    p = [Fraction(1, n)] * n
    assert gewinn(p) == gewinn_einfach(p)
print('Bis 8 Felder liefern beide Verfahren genau dasselbe.')

Rad aus der Aufgabe: 796/16875 = 0.04717037037037037
Bis 8 Felder liefern beide Verfahren genau dasselbe.


## Schritt 4: Testläufe für bis zu 10 Felder

Wie in den Lösungshinweisen probieren wir vier Räder mit jeweils 1 bis 10 Feldern aus:

1. alle Weiten gleich wahrscheinlich (`1/n` für jede Weite),
2. immer genau 1 Feld weiter (`p = (1, 0, 0, ...)`),
3. immer genau 3 Felder weiter (`p = (0, 0, 1, 0, ...)`),
4. 1 oder 2 Felder weiter, je mit Wahrscheinlichkeit 1/2.

In [ ]:
def rad_mit(n, werte):
    p = [Fraction(0)] * n
    for k in range(len(werte)):
        if k < n:
            p[k] = Fraction(werte[k])
    return p


print(' n    gleichverteilt    nur 1     nur 3    1 oder 2')
for n in range(1, 11):
    a = float(gewinn([Fraction(1, n)] * n))
    b = float(gewinn(rad_mit(n, [1])))
    c = float(gewinn(rad_mit(n, [0, 0, 1])))
    d = float(gewinn(rad_mit(n, [Fraction(1, 2), Fraction(1, 2)])))
    print(f'{n:2}  {a:14.6f} {b:8.0f} {c:8.0f} {d:11.6f}')

 n    gleichverteilt    nur 1     nur 3    1 oder 2
 1        1.000000        1        0    0.500000
 2        0.500000        1        0    0.500000
 3        0.222222        1        0    0.500000
 4        0.093750        1        1    0.250000
 5        0.038400        1        1    0.187500
 6        0.015432        1        0    0.093750
 7        0.006120        1        1    0.062500
 8        0.002403        1        1    0.031250
 9        0.000937        1        0    0.019531
10        0.000363        1        1    0.009766


(Für kleine `n` gibt es manche Weiten gar nicht: Bei 1 Feld gibt es nur die Weite 1, bei 1 oder 2 Feldern keine Weite 3. In diesen Zeilen ist die Tabelle unvollständig, die Wahrscheinlichkeiten ergeben nicht 1, und die Werte sind nicht aussagekräftig. Die Hinweise schreiben dort „n. a.“.)

Die Ergebnisse stimmen mit der Tabelle der Lösungshinweise überein, bis auf zwei Stellen, die dort offenbar Tippfehler sind: Für 3 Felder steht dort 0,25, richtig ist 6 · (1/3)³ = 0,2222, und für 9 Felder steht dort 0,00937, richtig ist
9! / 9⁹ ≈ 0,000937.
Unsere Zahlen kann man nachprüfen: Bei `n` Feldern mit jeweils der Wahrscheinlichkeit `1/n` führt **jede** der `n!` Reihenfolgen zum Gewinn, jede mit der Wahrscheinlichkeit `(1/n)ⁿ`, also ist das Ergebnis `n! / nⁿ`.

Außerdem sieht man: Wird immer nur 1 Feld weitergedreht, gewinnt man sicher (Wahrscheinlichkeit 1). Mit immer 3 Feldern gewinnt man nur, wenn 3 und `n` **teilerfremd** sind (bei 3, 6 und 9 Feldern kommt 0).
Das passt zur Mathematik: Man erreicht alle Felder genau dann, wenn die Schrittweite und die Feldzahl keinen gemeinsamen Teiler haben.

## Für Neugierige: Zehn Felder und Kontrolle

Das Rad mit 10 Feldern und den Wahrscheinlichkeiten 1/10, 2/10, 3/10 und 4/10 (und 0 für alle weiteren Weiten) als Beispiel mit ungleichen Wahrscheinlichkeiten. Außerdem eine Kontrolle durch **Simulation**:
Wir drehen das Rad 200 000-mal je `n` Mal, zählen, wie oft jedes Feld genau einmal erreicht wurde, und vergleichen den Anteil mit der berechneten Wahrscheinlichkeit.
`random.choices(weiten, weights=...)` zieht eine Schrittweite mit den gegebenen Wahrscheinlichkeiten.

In [ ]:
import random

p10 = rad_mit(10, [Fraction(1, 10), Fraction(2, 10), Fraction(3, 10), Fraction(4, 10)])
exakt = gewinn(p10)
print('Exakt berechnet:', float(exakt))


def simuliere(p, versuche):
    n = len(p)
    gewichte = [float(x) for x in p]
    gewonnen = 0
    for versuch in range(versuche):
        besucht = [False] * n
        pos = 0
        ok = True
        for dreh in range(n):
            k = random.choices(range(1, n + 1), weights=gewichte)[0]
            pos = (pos + k) % n
            if besucht[pos]:
                ok = False
                break
            besucht[pos] = True
        if ok:
            gewonnen = gewonnen + 1
    return gewonnen / versuche


random.seed(1)
print('Simuliert (200 000 Versuche):', simuliere(p10, 200000))
print('Rad aus der Aufgabe, simuliert:', simuliere(rad, 200000), ' exakt:', float(gewinn(rad)))

Exakt berechnet: 0.003038217
Simuliert (200 000 Versuche): 0.002795
Rad aus der Aufgabe, simuliert: 0.04812  exakt: 0.04717037037037037


Die Simulation liegt in der Nähe des berechneten Wertes. Bei sehr seltenen Ereignissen (wie 0,3 % beim ersten Rad) schwanken die simulierten Zahlen stärker, deshalb sind die Abweichungen dort größer als beim Rad aus der Aufgabe.
Das ist eine gute Kontrolle für die Rekursion, denn die Simulation spielt das Spiel einfach nach und kommt ohne Rekursion aus.